In [2]:
from ModifiedNEAT.base.initialization import initialize
from ModifiedNEAT.nn.base import NeatModule, NeatParameter
from ModifiedNEAT.nn.genome import Genome
from ModifiedNEAT.models.main import Transformer
from ModifiedNEAT.config import Config
from ModifiedNEAT.util.datetime import clock

import torch
import torch.nn as nn
import numpy as np
import warnings

In [3]:
# warnings.simplefilter('ignore', 'NumbaPerformanceWarning')

In [4]:
device = 'cuda'
dtype  = torch.float64

In [5]:
genomes = {key: Genome(key) for key in range(1, 101)}

In [6]:
batch_size  = 8
seq_len     = 256
input_dims  = 20
output_dims = 4
embedding   = 128
heads       = 4
layers      = 6
differential = True
bias        = True

In [7]:
constant = 2 ** np.floor(np.log2(seq_len * embedding)) // 2
constant

16384.0

In [8]:
def model_size(model: NeatModule):
    return np.sum([param.numel() * param.element_size() for param in model.parameters()]) / (1024 ** 3)

In [9]:
transformer = Transformer(input_dims, output_dims, embedding, seq_len, layers, heads, 1,
                          differential=differential, dropout=0.01, bias=bias,
                          constant=constant, causal_mask=True, device=device, dtype=dtype)

In [10]:
transformer.eval()
transformer.update(genomes)
transformer, model_size(transformer)

(Transformer(
   (primary_activation): SiLU()
   (embedder): BufferEmbedding(
     (embedding): Linear[NeatModule](inputs=20, outputs=128, bias=True)
   )
   (transformer): TransformerBase(
     (layers): ModuleList(
       (0): TransformerBlock(
         (attention): Attention(
           (query_proj): Linear[NeatModule](inputs=128, outputs=256, bias=True)
           (key_proj): Linear[NeatModule](inputs=128, outputs=64, bias=True)
           (value_proj): Linear[NeatModule](inputs=128, outputs=32, bias=True)
           (out_proj): Linear[NeatModule](inputs=128, outputs=128, bias=True)
           (rotary_embedding): RoPE()
           (softmax): Softmax(dim=-1)
           (norm): RMSNorm[NeatModule](shape=(32,), eps=1e-08, elementwise_affine=True)
           (diff_lambda): AttentionLambda[NeatModule](heads=4, head_dim=32, layer_idx=0)
         )
         (att_norm): RMSNorm[NeatModule](shape=(128,), eps=1e-08, elementwise_affine=True)
         (feedforward): SwiGLUFeedForward(
        

In [11]:
transformer.genome_num

100

In [12]:
transformer.neat_parameters()[0]

Parameter containing:
tensor([[[ 0.0641, -0.5999,  0.8500,  ..., -0.7435, -0.0350, -1.4040],
         [-0.5474, -0.7008, -0.9856,  ..., -1.1237,  0.0827, -0.6083],
         [ 0.3944, -1.2643,  1.9793,  ...,  0.1468,  0.2829, -0.3441],
         ...,
         [ 0.6606, -0.0771, -0.6857,  ..., -0.4124, -0.6984, -0.4190],
         [-0.7210, -1.2775,  0.7531,  ..., -0.1164,  0.0789,  1.4424],
         [ 0.8562,  1.4032, -0.0777,  ...,  0.1493, -2.0975,  1.8149]],

        [[ 0.0641, -0.5999,  0.8500,  ..., -0.7435, -0.0350, -1.4040],
         [-0.5474, -0.7008, -0.9856,  ..., -1.1237,  0.0827, -0.6083],
         [ 0.3944, -1.2643,  1.9793,  ...,  0.1468,  0.2829, -0.3441],
         ...,
         [ 0.6606, -0.0771, -0.6857,  ..., -0.4124, -0.6984, -0.4190],
         [-0.7210, -1.2775,  0.7531,  ..., -0.1164,  0.0789,  1.4424],
         [ 0.8562,  1.4032, -0.0777,  ...,  0.1493, -2.0975,  1.8149]],

        [[ 0.0641, -0.5999,  0.8500,  ..., -0.7435, -0.0350, -1.4040],
         [-0.5474, -0.7

In [13]:
config = Config()
config.genome.init_type = 'normal'
config.genome.init_type

'normal'

In [14]:
len(transformer.neat_parameters())

132

In [15]:
len(transformer.transformer.layers)

6

In [16]:
transformer.transformer.layers[-1].att_norm.weights.param_index

120

In [17]:
clone = [p.cpu().clone() for p in transformer.parameters()]

In [18]:
clone[0]

tensor([[[ 0.0641, -0.5999,  0.8500,  ..., -0.7435, -0.0350, -1.4040],
         [-0.5474, -0.7008, -0.9856,  ..., -1.1237,  0.0827, -0.6083],
         [ 0.3944, -1.2643,  1.9793,  ...,  0.1468,  0.2829, -0.3441],
         ...,
         [ 0.6606, -0.0771, -0.6857,  ..., -0.4124, -0.6984, -0.4190],
         [-0.7210, -1.2775,  0.7531,  ..., -0.1164,  0.0789,  1.4424],
         [ 0.8562,  1.4032, -0.0777,  ...,  0.1493, -2.0975,  1.8149]],

        [[ 0.0641, -0.5999,  0.8500,  ..., -0.7435, -0.0350, -1.4040],
         [-0.5474, -0.7008, -0.9856,  ..., -1.1237,  0.0827, -0.6083],
         [ 0.3944, -1.2643,  1.9793,  ...,  0.1468,  0.2829, -0.3441],
         ...,
         [ 0.6606, -0.0771, -0.6857,  ..., -0.4124, -0.6984, -0.4190],
         [-0.7210, -1.2775,  0.7531,  ..., -0.1164,  0.0789,  1.4424],
         [ 0.8562,  1.4032, -0.0777,  ...,  0.1493, -2.0975,  1.8149]],

        [[ 0.0641, -0.5999,  0.8500,  ..., -0.7435, -0.0350, -1.4040],
         [-0.5474, -0.7008, -0.9856,  ..., -1

In [19]:
ts = clock.perf_counter()
initialize(config, transformer, 10, 2)
round(clock.perf_counter() - ts)

(100, 20, 128)
torch.float64 cuda:0 ((10, 3, 20), (10, 10, 10)) 600000 torch.Size([100, 20, 128])
(100, 128, 1)
torch.float64 cuda:0 ((10, 20, 1), (10, 10, 10)) 200000 torch.Size([100, 128])
(100, 128, 256)
torch.float64 cuda:0 ((10, 20, 39), (10, 10, 10)) 7800000 torch.Size([100, 128, 256])
(100, 256, 1)
torch.float64 cuda:0 ((10, 39, 1), (10, 10, 10)) 390000 torch.Size([100, 256])
(100, 128, 64)
torch.float64 cuda:0 ((10, 20, 10), (10, 10, 10)) 2000000 torch.Size([100, 128, 64])
(100, 64, 1)
torch.float64 cuda:0 ((10, 10, 1), (10, 10, 10)) 100000 torch.Size([100, 64])
(100, 128, 32)
torch.float64 cuda:0 ((10, 20, 5), (10, 10, 10)) 1000000 torch.Size([100, 128, 32])
(100, 32, 1)
torch.float64 cuda:0 ((10, 5, 1), (10, 10, 10)) 50000 torch.Size([100, 32])


C:\Python3.11\Lib\site-packages\numba\cuda\dispatcher.py:536: NumbaPerformanceWarning: Grid size 100 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
C:\Python3.11\Lib\site-packages\numba\cuda\dispatcher.py:536: NumbaPerformanceWarning: Grid size 50 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


(100, 128, 128)
torch.float64 cuda:0 ((10, 20, 20), (10, 10, 10)) 4000000 torch.Size([100, 128, 128])
(100, 128, 1)
torch.float64 cuda:0 ((10, 20, 1), (10, 10, 10)) 200000 torch.Size([100, 128])
(100, 32, 1)
torch.float64 cuda:0 ((10, 5, 1), (10, 10, 10)) 50000 torch.Size([100, 32])
(100, 4, 32)
torch.float64 cuda:0 ((10, 1, 5), (10, 10, 10)) 50000 torch.Size([100, 4, 32])
(100, 4, 32)
torch.float64 cuda:0 ((10, 1, 5), (10, 10, 10)) 50000 torch.Size([100, 4, 32])
(100, 4, 32)
torch.float64 cuda:0 ((10, 1, 5), (10, 10, 10)) 50000 torch.Size([100, 4, 32])
(100, 4, 32)
torch.float64 cuda:0 ((10, 1, 5), (10, 10, 10)) 50000 torch.Size([100, 4, 32])
(100, 128, 1)
torch.float64 cuda:0 ((10, 20, 1), (10, 10, 10)) 200000 torch.Size([100, 128])
(100, 128, 512)
torch.float64 cuda:0 ((10, 20, 77), (10, 10, 10)) 15400000 torch.Size([100, 128, 512])
(100, 512, 1)
torch.float64 cuda:0 ((10, 77, 1), (10, 10, 10)) 770000 torch.Size([100, 512])
(100, 512, 128)
torch.float64 cuda:0 ((10, 77, 20), (10, 10

C:\Python3.11\Lib\site-packages\numba\cuda\dispatcher.py:536: NumbaPerformanceWarning: Grid size 10 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


69

In [20]:
for x, y in zip(clone, list(transformer.parameters())):
    print(x.shape, y.shape)
    error = torch.any(x.to(y.device) == y).cpu()
    x = x.cpu()
    y = y.cpu()
    if error:
        raise ValueError(f"A value in parameter was not randomized")

torch.Size([100, 20, 128]) torch.Size([100, 20, 128])
torch.Size([100, 128]) torch.Size([100, 128])
torch.Size([100, 128, 256]) torch.Size([100, 128, 256])
torch.Size([100, 256]) torch.Size([100, 256])
torch.Size([100, 128, 64]) torch.Size([100, 128, 64])
torch.Size([100, 64]) torch.Size([100, 64])
torch.Size([100, 128, 32]) torch.Size([100, 128, 32])
torch.Size([100, 32]) torch.Size([100, 32])
torch.Size([100, 128, 128]) torch.Size([100, 128, 128])
torch.Size([100, 128]) torch.Size([100, 128])
torch.Size([100, 32]) torch.Size([100, 32])
torch.Size([100, 4, 32]) torch.Size([100, 4, 32])
torch.Size([100, 4, 32]) torch.Size([100, 4, 32])
torch.Size([100, 4, 32]) torch.Size([100, 4, 32])
torch.Size([100, 4, 32]) torch.Size([100, 4, 32])
torch.Size([100, 128]) torch.Size([100, 128])
torch.Size([100, 128, 512]) torch.Size([100, 128, 512])
torch.Size([100, 512]) torch.Size([100, 512])
torch.Size([100, 512, 128]) torch.Size([100, 512, 128])
torch.Size([100, 128]) torch.Size([100, 128])
torch.

In [21]:
list(transformer.parameters())[0]

Parameter containing:
tensor([[[-2.3674e-01,  8.3971e-03,  7.7948e-03,  ...,  1.0091e-03,
           4.1993e-02, -1.3521e-01],
         [ 1.5827e-01, -2.7060e-01,  1.2796e-01,  ..., -1.5102e-01,
          -2.6711e-02, -4.4916e-02],
         [-9.0720e-05,  2.1779e-01, -1.0949e-01,  ..., -2.1962e-01,
           9.3276e-02,  2.1368e-02],
         ...,
         [ 1.1592e-01, -3.0238e-02,  3.7489e-02,  ...,  1.2639e-01,
           1.9455e-02, -1.6897e-01],
         [-2.6404e-02,  1.1620e-02, -4.1118e-02,  ..., -1.0811e-01,
          -1.6472e-01, -3.0241e-02],
         [ 1.1609e-02, -2.2636e-02,  6.4036e-02,  ..., -1.1374e-02,
          -2.4396e-03,  9.5223e-02]],

        [[-2.3814e-01, -1.6696e-01, -5.4285e-03,  ...,  9.9643e-02,
          -3.4701e-02,  4.6799e-02],
         [-1.0566e-01,  3.1058e-02,  1.4735e-01,  ...,  8.0425e-02,
           8.2321e-02,  4.0258e-02],
         [-2.2230e-02,  2.8626e-02,  1.1213e-01,  ...,  1.5979e-02,
          -1.4129e-01,  1.0396e-01],
         ...,
   

In [24]:
len(transformer.neat_parameters())

132